# 03 — Budget Optimizer & ROAS Curves

Goal: produce spend-response curves and an optimal allocation table.

**Input:** `data/processed/trace.nc`, `data/processed/clean.parquet`  
**Output:** `data/processed/roas_curves.parquet`, `data/processed/optimal_allocation.parquet`

> Run `02_model.ipynb` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

from src.mmm import BayesianMMM, BudgetOptimizer

PROC = Path('../data/processed')

CHANNEL_COLS = [
    'Paid_Views',
    'Google_Impressions',
    'Email_Impressions',
    'Facebook_Impressions',
    'Affiliate_Impressions',
]
TARGET_COL = 'Sales'

print('Setup complete.')

## 1. Load fitted model and data

In [ ]:
mmm = BayesianMMM.load(
    str(PROC / 'trace.nc'),
    channel_cols=CHANNEL_COLS,
    date_col='date',
    target_col=TARGET_COL,
)
opt = BudgetOptimizer(mmm)

df = pd.read_parquet(PROC / 'clean.parquet')
print('Model and data loaded.')

## 2. Current mean spend per channel

In [ ]:
# Use the mean across all weeks and divisions as the 'current' baseline
current_spend = df[CHANNEL_COLS].mean().values
total_budget  = current_spend.sum()

for ch, s in zip(CHANNEL_COLS, current_spend):
    print(f'{ch:<30} {s:>10,.1f}')
print(f'\nTotal budget: {total_budget:,.1f}')

## 3. ROAS / saturation curves per channel

In [ ]:
ncols = 3
nrows = -(-len(CHANNEL_COLS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))

all_curves = []
for ax, ch in zip(axes.flat, CHANNEL_COLS):
    # Sweep 0 → 2× current spend for this channel
    max_s = df[ch].mean() * 2
    spend_range = np.linspace(0, max_s, 300)
    curve = opt.roas_curve(ch, spend_range)
    curve['channel'] = ch
    all_curves.append(curve)

    ax.plot(curve['spend'], curve['response'], linewidth=2)
    ax.axvline(df[ch].mean(), color='red', linestyle='--', alpha=0.6, label='Mean spend')
    ax.set_title(ch, fontsize=10)
    ax.set_xlabel('Impressions / Views')
    ax.set_ylabel('Normalised Response')
    ax.legend(fontsize=8)

for ax in axes.flat[len(CHANNEL_COLS):]:
    ax.set_visible(False)

plt.suptitle('Saturation Curves (red dashed = mean spend)', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_roas_curves.png', bbox_inches='tight')

roas_df = pd.concat(all_curves, ignore_index=True)
roas_df.to_parquet(PROC / 'roas_curves.parquet', index=False)
print(f'ROAS curves saved → {len(roas_df):,} rows')

## 4. Budget optimisation

In [ ]:
result = opt.optimize(
    total_budget=float(total_budget),
    lower_bounds=[0.0] * len(CHANNEL_COLS),
    upper_bounds=[float(total_budget)] * len(CHANNEL_COLS),
    current_spend=current_spend,
)
print(result.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CHANNEL_COLS))
width = 0.35
ax.bar(x - width/2, result['current_spend'], width, label='Current')
ax.bar(x + width/2, result['optimal_spend'], width, label='Optimal')
ax.set_xticks(x)
ax.set_xticklabels(CHANNEL_COLS, rotation=20, ha='right')
ax.set_ylabel('Mean impressions / views')
ax.set_title('Current vs Optimal Spend Allocation')
ax.legend()
plt.tight_layout()
fig.savefig(PROC / 'fig_optimal_allocation.png', bbox_inches='tight')

result.to_parquet(PROC / 'optimal_allocation.parquet', index=False)
print('Optimal allocation saved → data/processed/optimal_allocation.parquet')